In [25]:
import pandas as pd
import numpy as np
import os

import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../../data/raw/banksim.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 594642 entries, 0 to 594641
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   step         594642 non-null  int64  
 1   customer     594642 non-null  str    
 2   age          594642 non-null  str    
 3   gender       594642 non-null  str    
 4   zipcodeori   594642 non-null  str    
 5   merchant     594642 non-null  str    
 6   zipmerchant  594642 non-null  str    
 7   category     594642 non-null  str    
 8   amount       594642 non-null  float64
 9   fraud        594642 non-null  int64  
dtypes: float64(1), int64(2), str(7)
memory usage: 81.2 MB


In [4]:
df.describe()

,step,amount,fraud
count,594642.000000,594642.000000,594642.000000
mean,94.986987,37.890191,0.012108
std,51.053526,111.402916,0.109369
min,0.000000,0.000000,0.000000
25%,52.000000,13.740000,0.000000
50%,97.000000,26.900000,0.000000
75%,139.000000,42.540000,0.000000
max,179.000000,8329.960000,1.000000


In [5]:
df.isna().sum()

step           0
customer       0
age            0
gender         0
zipcodeori     0
merchant       0
zipmerchant    0
category       0
amount         0
fraud          0
dtype: int64

In [6]:
df.duplicated().any()

np.False_

In [7]:
df[df.duplicated()]

,step,customer,age,gender,zipcodeori,merchant,zipmerchant,category,amount,fraud


### Sort columns into their data type

In [8]:
cat_cols, cont_cols = [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        cont_cols.append(col)

print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)

Categorical Columns: ['customer', 'age', 'gender', 'zipcodeori', 'merchant', 'zipmerchant', 'category']
Continuous Columns: ['step', 'amount', 'fraud']


### Check for columns with no variability
If they have no variance they have no use to a model due to thier features being constant

In [9]:
cols_to_drop = []
for col in cat_cols:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

Unique numbers of customers: 4112
Unique numbers of ages: 8
Unique numbers of genders: 4
Unique numbers of zipcodeoris: 1
Unique numbers of merchants: 50
Unique numbers of zipmerchants: 1
Unique numbers of categorys: 15

Dropping columns due to constant values: ['zipcodeori', 'zipmerchant']


### Extract Customer-based Features (All local features)
These include: 

- "prev_step" - The last time they did a transaction (-1 if no previous transaction exists)
- "time_since_last_transaction" --
- "transaction_count" - Number of transactions

- "amount_sum" - Sum of all previous transactions
- "avg_amount" - Avergae of all past transactions
- "std_amount" - Standard deviation of past transaction amounts
- "amount_zscore" - How many standard deviation this transaction is from past behaviour

- "avg_amount_ratio" - Ratio of how this given transaction compares to customers average
- "log_amount_ratio" - Log transformation to compress extreme values

- "merchant_count" - Number of times a customer has shopped with this merchant
- "category_count" - Number of times a customer has shopped this category
- "prev_merchant_step" - Last time they shopped with this merchant (-1 if they never have)
- "time_since_last_merchant_transaction" --

In [10]:
# Just checking if the rolling features are working as expected (NEED TO REVERT THIS SORTING BACK TO ORIGINAL ORDER LATER)
df = df.sort_values(["customer", "step"]).copy()
df.rename(columns={"amount": "customer_amount"}, inplace=True)

cust_groups = df.groupby("customer")

# Shift gets the previous step for each customer
df["customer_prev_step"] = cust_groups["step"].shift(1)
df["customer_time_since_last_transaction"] = df["step"] - df["customer_prev_step"]

df["customer_time_since_last_transaction"] = df["customer_time_since_last_transaction"].fillna(-1)
df["customer_prev_step"] = df["customer_prev_step"].fillna(-1)

df["customer_transaction_count"] = cust_groups.cumcount()

df["customer_amount_sum"] = cust_groups["customer_amount"].cumsum() - df["customer_amount"] 
df["customer_amount_sum"] = df["customer_amount_sum"].fillna(0)

df["customer_avg_amount"] = df["customer_amount_sum"] / df["customer_transaction_count"]
df["customer_avg_amount"] = df["customer_avg_amount"].fillna(df["customer_amount"])

# Expanding std gives the std of all previous transactions, shift(1) makes it so that the current transaction is not included in the std calculation
df["customer_std_amount"] = cust_groups["customer_amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["customer_std_amount"] = df["customer_std_amount"].fillna(0)

df["customer_avg_amount_ratio"] = df["customer_amount"] / df["customer_avg_amount"]
df["customer_log_amount_ratio"] = np.log1p(df["customer_avg_amount_ratio"])

df["customer_zscore"] = (df["customer_amount"] - df["customer_avg_amount"]) / df["customer_std_amount"]
df["customer_zscore"] = df["customer_zscore"].fillna(0)
df["customer_zscore"] = df["customer_zscore"].replace([np.inf, -np.inf], 0)

df["customer_merchant_count"] = df.groupby(["customer", "merchant"]).cumcount()
df["customer_category_count"] = df.groupby(["customer", "category"]).cumcount()

df["customer_prev_merchant_step"] = df.groupby(["customer", "merchant"])["step"].shift(1)
df["customer_time_since_last_merchant_transaction"] = df["step"] - df["customer_prev_merchant_step"]

df["customer_time_since_last_merchant_transaction"] = df["customer_time_since_last_merchant_transaction"].fillna(-1)
df["customer_prev_merchant_step"] = df["customer_prev_merchant_step"].fillna(-1)

df

,step,customer,age,gender,merchant,category,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,...,customer_amount_sum,customer_avg_amount,customer_std_amount,customer_avg_amount_ratio,customer_log_amount_ratio,customer_zscore,customer_merchant_count,customer_category_count,customer_prev_merchant_step,customer_time_since_last_merchant_transaction
80562,30,'C1000148617','5','M','M1888755466','es_otherservices',143.87,0,-1.0,-1.0,...,0.00,143.870000,0.000000,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
105385,38,'C1000148617','5','M','M1741626453','es_sportsandtoys',16.69,0,30.0,8.0,...,143.87,143.870000,0.000000,0.116008,0.109758,0.000000,0,0,-1.0,-1.0
117325,42,'C1000148617','5','M','M1888755466','es_otherservices',56.18,0,38.0,4.0,...,160.56,80.280000,89.929840,0.699801,0.530511,-0.267987,1,1,30.0,12.0
120413,43,'C1000148617','5','M','M840466850','es_tech',14.74,0,42.0,1.0,...,216.74,72.246667,65.094481,0.204023,0.185669,-0.883434,0,0,-1.0,-1.0
124901,44,'C1000148617','5','M','M1823072687','es_transportation',47.42,0,43.0,1.0,...,231.48,57.870000,60.428595,0.819423,0.598519,-0.172931,0,0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573564,174,'C999723254','2','M','M1823072687','es_transportation',31.94,0,173.0,1.0,...,3405.12,29.103590,35.578612,1.097459,0.740727,0.079722,73,100,173.0,1.0
581160,176,'C999723254','2','M','M1823072687','es_transportation',1.92,0,174.0,2.0,...,3437.06,29.127627,35.427202,0.065917,0.063835,-0.767987,74,101,174.0,2.0
585495,177,'C999723254','2','M','M85975013','es_food',62.55,0,176.0,1.0,...,3438.98,28.898992,35.364827,2.164435,1.151975,0.951539,7,7,170.0,7.0
587473,178,'C999723254','2','M','M1823072687','es_transportation',25.96,0,177.0,1.0,...,3501.53,29.179417,35.349650,0.889668,0.636401,-0.091074,75,102,176.0,2.0


In [11]:
# Sort back to original dataset
df = df.sort_index()
df

,step,customer,age,gender,merchant,category,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,...,customer_amount_sum,customer_avg_amount,customer_std_amount,customer_avg_amount_ratio,customer_log_amount_ratio,customer_zscore,customer_merchant_count,customer_category_count,customer_prev_merchant_step,customer_time_since_last_merchant_transaction
0,0,'C352968107','2','M','M348934600','es_transportation',39.68,0,-1.0,-1.0,...,0.00,39.680000,28.754816,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
1,0,'C2054744914','4','F','M1823072687','es_transportation',26.89,0,-1.0,-1.0,...,0.00,26.890000,31.006308,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
2,0,'C1760612790','3','M','M348934600','es_transportation',17.25,0,-1.0,-1.0,...,0.00,17.250000,70.442508,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
3,0,'C757503768','5','M','M348934600','es_transportation',35.72,0,-1.0,-1.0,...,0.00,35.720000,29.222879,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
4,0,'C1315400589','3','F','M348934600','es_transportation',25.81,0,-1.0,-1.0,...,0.00,25.810000,26.273975,1.000000,0.693147,0.000000,0,0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594637,179,'C1753498738','3','F','M1823072687','es_transportation',20.53,0,178.0,1.0,...,6049.13,33.055355,33.666381,0.621079,0.483092,-0.372043,154,163,178.0,1.0
594638,179,'C650108285','4','F','M1823072687','es_transportation',50.73,0,178.0,1.0,...,5988.40,35.019883,29.322665,1.448606,0.895519,0.535767,109,144,178.0,1.0
594639,179,'C123623130','2','F','M349281107','es_fashion',22.44,0,178.0,1.0,...,5231.59,31.326886,24.374884,0.716318,0.540181,-0.364592,0,0,-1.0,-1.0
594640,179,'C1499363341','5','M','M1823072687','es_transportation',14.46,0,178.0,1.0,...,4606.05,28.085671,18.827604,0.514853,0.415319,-0.723707,96,147,178.0,1.0


### Extract Merchant-based features (ALl local features)
All of these are the same as customer based features execept for:
- "merchant_fraud_count" - How many times has fraud occured for this merchant
- "merchant_fraud_rate" - Rate of fraud up to this transaction

In [12]:
df = df.sort_values(["merchant", "step"]).copy()
merchant_groups = df.groupby("merchant")

df["merchant_transaction_count"] = merchant_groups.cumcount()

df["merchant_past_amount_sum"] = merchant_groups["customer_amount"].cumsum() - df["customer_amount"]
df["merchant_past_amount_sum"] = df["merchant_past_amount_sum"].fillna(0)

df["merchant_avg_amount"] = df["merchant_past_amount_sum"] / df["merchant_transaction_count"]
df["merchant_avg_amount"] = df["merchant_avg_amount"].fillna(df["customer_amount"])

df["merchant_std_amount"] = merchant_groups["customer_amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["merchant_std_amount"] = df["merchant_std_amount"].fillna(0)

df["merchant_avg_amount_ratio"] = df["customer_amount"] / df["merchant_avg_amount"]
df["merchant_log_amount_ratio"] = np.log1p(df["merchant_avg_amount_ratio"])

df["merchant_amount_zscore"] = (df["customer_amount"] - df["merchant_avg_amount"]) / df["merchant_std_amount"]
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].fillna(0)
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].replace([np.inf, -np.inf], 0)

df["merchant_fraud_count"] = merchant_groups["fraud"].cumsum() - df["fraud"]
df["merchant_fraud_rate"] = df["merchant_fraud_count"] / df["merchant_transaction_count"]
df["merchant_fraud_rate"] = df["merchant_fraud_rate"].fillna(0)

df["merchant_prev_step"] = merchant_groups["step"].shift(1)
df["merchant_time_since_last_transaction"] = df["step"] - df["merchant_prev_step"]

df["merchant_time_since_last_transaction"] = df["merchant_time_since_last_transaction"].fillna(-1)
df["merchant_prev_step"] = df["merchant_prev_step"].fillna(-1)

df


,step,customer,age,gender,merchant,category,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,...,merchant_past_amount_sum,merchant_avg_amount,merchant_std_amount,merchant_avg_amount_ratio,merchant_log_amount_ratio,merchant_amount_zscore,merchant_fraud_count,merchant_fraud_rate,merchant_prev_step,merchant_time_since_last_transaction
41,0,'C1635613216','4','F','M1053599405','es_health',105.59,0,-1.0,-1.0,...,0.00,105.590000,0.000000,1.000000,0.693147,0.000000,0,0.000000,-1.0,-1.0
78,0,'C118437987','2','M','M1053599405','es_health',159.92,0,-1.0,-1.0,...,105.59,105.590000,0.000000,1.514537,0.922089,0.000000,0,0.000000,0.0,0.0
130,0,'C650108285','4','F','M1053599405','es_health',11.83,0,-1.0,-1.0,...,265.51,132.755000,38.417111,0.089112,0.085362,-3.147686,0,0.000000,0.0,0.0
160,0,'C1463833315','1','M','M1053599405','es_health',7.94,0,-1.0,-1.0,...,277.34,92.446667,74.914768,0.085887,0.082397,-1.128037,0,0.000000,0.0,0.0
550,0,'C1810630647','2','F','M1053599405','es_health',105.54,0,-1.0,-1.0,...,285.28,71.320000,74.342624,1.479809,0.908182,0.460301,0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591304,179,'C1647495093','3','M','M980657600','es_sportsandtoys',396.66,1,178.0,1.0,...,528733.84,299.735737,238.787331,1.323366,0.843017,0.405902,1467,0.831633,179.0,0.0
591583,179,'C910454738','4','F','M980657600','es_sportsandtoys',203.18,1,177.0,2.0,...,529130.50,299.790652,238.730786,0.677740,0.517447,-0.404685,1468,0.831728,179.0,0.0
591584,179,'C2015444413','2','F','M980657600','es_sportsandtoys',559.31,1,178.0,1.0,...,529333.68,299.735946,238.674220,1.866009,1.052921,1.087566,1469,0.831823,179.0,0.0
591897,179,'C881073190','2','F','M980657600','es_sportsandtoys',224.20,1,177.0,2.0,...,529892.99,299.882847,238.686527,0.747625,0.558258,-0.317081,1470,0.831919,179.0,0.0


In [13]:
# Sort back to original dataset
df = df.sort_index()
df

,step,customer,age,gender,merchant,category,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,...,merchant_past_amount_sum,merchant_avg_amount,merchant_std_amount,merchant_avg_amount_ratio,merchant_log_amount_ratio,merchant_amount_zscore,merchant_fraud_count,merchant_fraud_rate,merchant_prev_step,merchant_time_since_last_transaction
0,0,'C352968107','2','M','M348934600','es_transportation',39.68,0,-1.0,-1.0,...,0.00,39.680000,93.836179,1.000000,0.693147,0.000000,0,0.0,-1.0,-1.0
1,0,'C2054744914','4','F','M1823072687','es_transportation',26.89,0,-1.0,-1.0,...,0.00,26.890000,62.383912,1.000000,0.693147,0.000000,0,0.0,-1.0,-1.0
2,0,'C1760612790','3','M','M348934600','es_transportation',17.25,0,-1.0,-1.0,...,39.68,39.680000,0.000000,0.434728,0.360975,0.000000,0,0.0,0.0,0.0
3,0,'C757503768','5','M','M348934600','es_transportation',35.72,0,-1.0,-1.0,...,56.93,28.465000,15.860405,1.254874,0.813094,0.457428,0,0.0,0.0,0.0
4,0,'C1315400589','3','F','M348934600','es_transportation',25.81,0,-1.0,-1.0,...,92.65,30.883333,11.971685,0.835726,0.607440,-0.423778,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594637,179,'C1753498738','3','F','M1823072687','es_transportation',20.53,0,178.0,1.0,...,8077556.41,26.953129,17.528902,0.761693,0.566275,-0.366431,0,0.0,179.0,0.0
594638,179,'C650108285','4','F','M1823072687','es_transportation',50.73,0,178.0,1.0,...,8077576.94,26.953108,17.528876,1.882158,1.058539,1.356441,0,0.0,179.0,0.0
594639,179,'C123623130','2','F','M349281107','es_fashion',22.44,0,178.0,1.0,...,178405.68,61.946417,44.364135,0.362249,0.309137,-0.890503,0,0.0,179.0,0.0
594640,179,'C1499363341','5','M','M1823072687','es_transportation',14.46,0,178.0,1.0,...,8077627.67,26.953187,17.528901,0.536486,0.429498,-0.712719,0,0.0,179.0,0.0


### Extract Global Features
Features are the same as both customer and merchant just no longer localised features.

In [14]:
df["global_transaction_count"] = df.index

df["global_amount_sum"] = df["customer_amount"].cumsum() - df["customer_amount"]
df["global_amount_sum"] = df["global_amount_sum"].fillna(0)

df["global_avg_amount"] = df["global_amount_sum"] / df["global_transaction_count"]
df["global_avg_amount"] = df["global_avg_amount"].fillna(df["customer_amount"])

df["global_amount_ratio"] = df["customer_amount"] / df["global_avg_amount"]
df["global_log_amount_ratio"] = np.log1p(df["global_amount_ratio"])

df["global_std_amount"] = df["customer_amount"].expanding().std().shift(1).reset_index(drop=True)
df["global_std_amount"] = df["global_std_amount"].fillna(0)

df["global_z_score"] = (df["customer_amount"] - df["global_avg_amount"]) / df["global_std_amount"]
df["global_z_score"] = df["global_z_score"].fillna(0)
df["global_z_score"] = df["global_z_score"].replace([np.inf, -np.inf], 0)

df["global_median_amount"] = df["customer_amount"].expanding().median().shift(1).reset_index(drop=True)
df["global_median_amount"] = df["global_median_amount"].fillna(df["customer_amount"])

df["global_median_amount_ratio"] = df["customer_amount"] / df["global_median_amount"]
df["global_log_median_amount_ratio"] = np.log1p(df["global_median_amount_ratio"])

df.drop(columns="global_transaction_count", inplace=True)
df

,step,customer,age,gender,merchant,category,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,...,merchant_time_since_last_transaction,global_amount_sum,global_avg_amount,global_amount_ratio,global_log_amount_ratio,global_std_amount,global_z_score,global_median_amount,global_median_amount_ratio,global_log_median_amount_ratio
0,0,'C352968107','2','M','M348934600','es_transportation',39.68,0,-1.0,-1.0,...,-1.0,0.00,39.680000,1.000000,0.693147,0.000000,0.000000,39.680,1.000000,0.693147
1,0,'C2054744914','4','F','M1823072687','es_transportation',26.89,0,-1.0,-1.0,...,-1.0,39.68,39.680000,0.677671,0.517407,0.000000,0.000000,39.680,0.677671,0.517407
2,0,'C1760612790','3','M','M348934600','es_transportation',17.25,0,-1.0,-1.0,...,0.0,66.57,33.285000,0.518251,0.417559,9.043896,-1.773019,33.285,0.518251,0.417559
3,0,'C757503768','5','M','M348934600','es_transportation',35.72,0,-1.0,-1.0,...,0.0,83.82,27.940000,1.278454,0.823497,11.251804,0.691445,26.890,1.328375,0.845171
4,0,'C1315400589','3','F','M348934600','es_transportation',25.81,0,-1.0,-1.0,...,0.0,119.54,29.885000,0.863644,0.622534,9.976681,-0.408452,31.305,0.824469,0.601289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594637,179,'C1753498738','3','F','M1823072687','es_transportation',20.53,0,178.0,1.0,...,0.0,22530964.09,37.890283,0.541828,0.432968,111.403374,-0.155833,26.900,0.763197,0.567129
594638,179,'C650108285','4','F','M1823072687','es_transportation',50.73,0,178.0,1.0,...,0.0,22530984.62,37.890254,1.338867,0.849667,111.403283,0.115255,26.900,1.885874,1.059828
594639,179,'C123623130','2','F','M349281107','es_fashion',22.44,0,178.0,1.0,...,0.0,22531035.35,37.890275,0.592236,0.465140,111.403190,-0.138688,26.900,0.834201,0.606609
594640,179,'C1499363341','5','M','M1823072687','es_transportation',14.46,0,178.0,1.0,...,0.0,22531057.79,37.890249,0.381629,0.323263,111.403099,-0.210320,26.900,0.537546,0.430188


In [15]:
df.describe()

,step,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,customer_transaction_count,customer_amount_sum,customer_avg_amount,customer_std_amount,customer_avg_amount_ratio,...,merchant_time_since_last_transaction,global_amount_sum,global_avg_amount,global_amount_ratio,global_log_amount_ratio,global_std_amount,global_z_score,global_median_amount,global_median_amount_ratio,global_log_median_amount_ratio
count,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,...,594642.000000,5.946420e+05,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000,594642.000000
mean,94.986987,37.890191,0.012108,93.749162,1.190967,78.134291,2899.365717,41.641807,54.364419,1.042525,...,0.014684,1.141503e+07,38.802497,0.976512,0.559187,120.559642,-0.007338,26.929430,1.406987,0.716450
std,51.053526,111.402916,0.109369,51.193850,2.027126,48.305450,2992.417530,58.110890,118.307007,3.157158,...,0.225739,6.477454e+06,0.857763,2.863187,0.384535,10.010603,0.928032,0.073872,4.136016,0.449711
min,0.000000,0.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.050000,0.000000,0.000000,...,-1.000000,0.000000e+00,23.625455,0.000000,0.000000,0.000000,-2.407281,21.190000,0.000000,0.000000
25%,52.000000,13.740000,0.000000,51.000000,1.000000,36.000000,1280.472500,28.850000,20.551484,0.374968,...,0.000000,5.845583e+06,38.131246,0.353821,0.302931,113.945033,-0.207905,26.890000,0.510227,0.412260
50%,97.000000,26.900000,0.000000,96.000000,1.000000,75.000000,2582.755000,31.583085,25.961661,0.782327,...,0.000000,1.145931e+07,38.541527,0.693619,0.526868,117.849011,-0.098702,26.930000,0.998886,0.692590
75%,139.000000,42.540000,0.000000,138.000000,1.000000,118.000000,3973.487500,36.317553,37.653688,1.288560,...,0.000000,1.700683e+07,39.315711,1.097065,0.740539,126.722294,0.031305,26.950000,1.579926,0.947761
max,179.000000,8329.960000,1.000000,179.000000,116.000000,264.000000,83740.370000,6888.300000,4834.906256,1108.062500,...,97.000000,2.253107e+07,42.358908,218.848522,5.392939,163.393701,79.374222,39.680000,309.663941,5.738712


### Hot-Encode values
Converts categorical data such as age and data into binary so the model can interpret them

In [16]:
# Removes leading/trailing quotes from categorical columns
for col in ['customer', 'merchant', 'category']:
    df[col] = df[col].astype(str).str.replace("'", "")

In [17]:
df = pd.get_dummies(df, columns=["age", "gender", "category"])

# Just incase it writes back as string when reading the dataset
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

df

,step,customer,merchant,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,customer_transaction_count,customer_amount_sum,customer_avg_amount,...,category_es_home,category_es_hotelservices,category_es_hyper,category_es_leisure,category_es_otherservices,category_es_sportsandtoys,category_es_tech,category_es_transportation,category_es_travel,category_es_wellnessandbeauty
0,0,C352968107,M348934600,39.68,0,-1.0,-1.0,0,0.00,39.680000,...,0,0,0,0,0,0,0,1,0,0
1,0,C2054744914,M1823072687,26.89,0,-1.0,-1.0,0,0.00,26.890000,...,0,0,0,0,0,0,0,1,0,0
2,0,C1760612790,M348934600,17.25,0,-1.0,-1.0,0,0.00,17.250000,...,0,0,0,0,0,0,0,1,0,0
3,0,C757503768,M348934600,35.72,0,-1.0,-1.0,0,0.00,35.720000,...,0,0,0,0,0,0,0,1,0,0
4,0,C1315400589,M348934600,25.81,0,-1.0,-1.0,0,0.00,25.810000,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594637,179,C1753498738,M1823072687,20.53,0,178.0,1.0,183,6049.13,33.055355,...,0,0,0,0,0,0,0,1,0,0
594638,179,C650108285,M1823072687,50.73,0,178.0,1.0,171,5988.40,35.019883,...,0,0,0,0,0,0,0,1,0,0
594639,179,C123623130,M349281107,22.44,0,178.0,1.0,167,5231.59,31.326886,...,0,0,0,0,0,0,0,0,0,0
594640,179,C1499363341,M1823072687,14.46,0,178.0,1.0,164,4606.05,28.085671,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
os.makedirs("../../data/processed", exist_ok=True)
df.to_csv("../../data/processed/processed_data.csv", index=False)

### Reduce Dimensions of the dataset
Drop features for reasons such as:
- Being used to derive other more useful features.
- Better signal for the models.
- Data analysis pointed towards dropping it.

Overall for the base DescisionTree it has reduced False Negatives from 77 -> 47 but raised False Positives from 674 -> 1468,
and for the XGBoost model it raised it from 25 -> 28 but also dropped False Positives from 632 -> 586.

From the permutation and importance graphs of both models a reason for some of these changes such as False Posiitves within the DecisionTree is when I removed some unnesscary global features it was harder for the model to distibguihs rules as local values are more noisy for tree values, XGBoost model imporves this as it uses more local features compared to the global features.

In [18]:
# Time based columns as they are arbitrary as time grows and time_since_last_transactions captures better meaning
df.drop(columns=["customer_prev_step", "customer_prev_merchant_step", "merchant_prev_step"], inplace=True)

# Sums and averages are captured in the ratios and z-scores, so can drop them to reduce dimensionality
df.drop(columns=["customer_amount_sum", "customer_avg_amount", "merchant_past_amount_sum", "merchant_avg_amount", "global_amount_sum"], inplace=True)

# Was only used to dervive fraud rate, can drop it now
df.drop(columns=["merchant_fraud_count"], inplace=True)

# Dropping from data analyisis, median captures more of the more common transaction amounts, compared to the mean
df.drop(columns=["global_avg_amount", "global_amount_ratio", "global_log_amount_ratio", "global_median_amount"], inplace=True)

# From Permutation graphs log values tend to do better
df.drop(columns=["customer_avg_amount_ratio", "merchant_avg_amount_ratio", "global_median_amount_ratio"], inplace=True)

In [20]:
# Columns to standardise for non tree based models
scale_cols = [
    "customer_amount",
    "customer_time_since_last_transaction",
    "customer_transaction_count",
    "customer_std_amount",
    "customer_merchant_count",
    "customer_category_count",
    "customer_time_since_last_merchant_transaction",
    "merchant_transaction_count",
    "merchant_std_amount",
    "merchant_time_since_last_transaction",
    "global_std_amount"  
]

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), scale_cols)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# save processor for future inputs
joblib.dump(preprocessor, "../../data/processed/preprocessor.pkl")

['../../data/processed/preprocessor.pkl']

### Train/Val/Test split
Creates a train/val/test split of 70/15/15 and standardises and transforms each set

In [22]:
split_step = df["step"].quantile(0.7)
val_step = df["step"].quantile(0.85)

train_df = df[df["step"] <= split_step]
val_df = df[(df["step"] > split_step) & (df["step"] <= val_step)]
test_df = df[df["step"] > val_step]

train_df.drop(columns=["step"], inplace=True)
val_df.drop(columns=["step"], inplace=True)
test_df.drop(columns=["step"], inplace=True)

train_scaled = preprocessor.fit_transform(train_df)
val_scaled = preprocessor.transform(val_df)
test_scaled = preprocessor.transform(test_df)

cols = preprocessor.get_feature_names_out()

train_scaled = pd.DataFrame(train_scaled, columns=cols)
val_scaled = pd.DataFrame(val_scaled, columns=cols)
test_scaled = pd.DataFrame(test_scaled, columns=cols)

train_scaled

,customer_amount,customer_time_since_last_transaction,customer_transaction_count,customer_std_amount,customer_merchant_count,customer_category_count,customer_time_since_last_merchant_transaction,merchant_transaction_count,merchant_std_amount,merchant_time_since_last_transaction,...,category_es_home,category_es_hotelservices,category_es_hyper,category_es_leisure,category_es_otherservices,category_es_sportsandtoys,category_es_tech,category_es_transportation,category_es_travel,category_es_wellnessandbeauty
0,0.012815,-1.029596,-1.598592,-0.183004,-1.116318,-1.279647,-0.434127,-1.342489,0.934218,-4.150338,...,0,0,0,0,0,0,0,1,0,0
1,-0.098304,-1.029596,-1.598592,-0.164278,-1.116318,-1.279647,-0.434127,-1.342489,0.48999,-4.150338,...,0,0,0,0,0,0,0,1,0,0
2,-0.182055,-1.029596,-1.598592,0.163729,-1.116318,-1.279647,-0.434127,-1.342471,-0.391113,-0.062639,...,0,0,0,0,0,0,0,1,0,0
3,-0.021589,-1.029596,-1.598592,-0.179111,-1.116318,-1.279647,-0.434127,-1.342454,-0.167103,-0.062639,...,0,0,0,0,0,0,0,1,0,0
4,-0.107686,-1.029596,-1.598592,-0.203639,-1.116318,-1.279647,-0.434127,-1.342436,-0.222027,-0.062639,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417842,0.136357,-0.101164,1.697568,-0.126194,1.346231,1.684068,-0.197814,1.604694,-0.143466,-0.062639,...,0,0,0,0,0,0,0,1,0,0
417843,-0.19917,-0.101164,2.063808,-0.186975,1.458165,2.252861,-0.197814,1.966835,-0.143772,-0.062639,...,0,0,0,0,0,0,0,1,0,0
417844,-0.201168,-0.101164,1.922946,-0.217373,2.055147,1.893623,-0.197814,1.966853,-0.143772,-0.062639,...,0,0,0,0,0,0,0,1,0,0
417845,0.031841,-0.101164,1.556706,-0.161779,1.86859,1.444576,-0.197814,1.966871,-0.143772,-0.062639,...,0,0,0,0,0,0,0,1,0,0


In [23]:
os.makedirs("../../results/models", exist_ok=True)

os.makedirs("../../data/processed/supervised", exist_ok=True)
os.makedirs("../../data/processed/unsupervised", exist_ok=True)

cols_to_drop = ["customer", "merchant"] + [col for col in train_scaled.columns if col.startswith("category")]

supervised_train_df = train_scaled.copy()
supervised_train_df.drop(columns=cols_to_drop, inplace=True)
supervised_train_df.to_csv("../../data/processed/supervised/train_data.csv", index=False)

supervised_val_df = val_scaled.copy()
supervised_val_df.drop(columns=cols_to_drop, inplace=True)
supervised_val_df.to_csv("../../data/processed/supervised/val_data.csv", index=False)

supervised_test_df = test_scaled.copy()
supervised_test_df.drop(columns=cols_to_drop, inplace=True)
supervised_test_df.to_csv("../../data/processed/supervised/test_data.csv", index=False)


In [24]:
unsupervised_train_df = train_scaled.copy()
unsupervised_train_df = unsupervised_train_df[unsupervised_train_df["fraud"] == 0]
unsupervised_train_df.drop(columns=["fraud"], inplace=True)
unsupervised_train_df.to_csv("../../data/processed/unsupervised/train_data.csv", index=False)

unsupervised_val_df = val_scaled.copy()
unsupervised_val_df.to_csv("../../data/processed/unsupervised/val_data.csv", index=False)

unsupervised_test_df = test_scaled.copy()
unsupervised_test_df.to_csv("../../data/processed/unsupervised/test_data.csv", index=False)